# sim8 — интерактивный симулятор в Jupyter

## Настройка (один раз)

```bash
cd pysim8
uv add --dev ipykernel
uv run python -m ipykernel install --user --name sim8 --display-name "sim8 assembly"
```

Открыть notebook: **Kernel → Change Kernel → sim8 assembly**

## Magic-директивы

| Magic | Действие |
|---|---|
| `%arch 1/2/3` | Установить архитектуру (1=int, 2=+FPU, 3=+VU), сбросить sim |
| `%page N` | Показать страницу памяти N (256 байт) после запуска |
| `%mem ADDR [N]` | Показать N байт по адресу ADDR (hex `0x...` или dec, по умолчанию N=32) |
| `%trace` | Показать трассировку после запуска |
| `%reset` | Сбросить sim без запуска кода |

По умолчанию: **arch=3** (целые + FPU + VU).

## 1. Базовые целочисленные операции (arch=1)

Каждая ячейка — asm-код. Результат (регистры, флаги, I/O) отображается автоматически.

In [4]:
%arch 3
        MOV A, 10
        MOV B, 7
        SUB A, B    ; A = 3
        ADD A, 5    ; A = 8
        MOV C, A    ; C = 8

CPU · HALTED · 5 steps · 5 cycles
  A=0x08  B=0x07  C=0x08  D=0x00  SP=0xE7  DP=0x00  IP=0x0F
  FLAGS: —
  FA=0x00000000  FB=0x00000000  FPCR=0x00  FPSR=0x00
  VA=0x0000  VB=0x0000  VC=0x0000  VM=0x0000  VL=0  VFPSR=0x00

## 2. Трассировка выполнения

`%trace` добавляет таблицу шагов под результатом.

In [8]:
%arch 3
%trace
        MOV A, 1
        MOV B, 1
        ADD A, B    ; A = 2
        ADD A, B    ; A = 3
        ADD A, B    ; A = 4
        HLT

CPU · HALTED · 5 steps · 5 cycles
  A=0x04  B=0x01  C=0x00  D=0x00  SP=0xE7  DP=0x00  IP=0x0F
  FLAGS: —
  FA=0x00000000  FB=0x00000000  FPCR=0x00  FPSR=0x00
  VA=0x0000  VB=0x0000  VC=0x0000  VM=0x0000  VL=0  VFPSR=0x00

#,IP,Opcode,Operands,Changes,Cost,Fault
0,0,6,0x00 0x01,"A=1, IP=3",1,
1,3,6,0x01 0x01,"B=1, IP=6",1,
2,6,10,0x00 0x01,"A=2, IP=9",1,
3,9,10,0x00 0x01,"A=3, IP=12",1,
4,12,10,0x00 0x01,"A=4, IP=15",1,


## 3. Дамп памяти

`%page N` показывает hex-дамп страницы N (256 байт). Страница 0: код + I/O-регион (адреса 232–255).

In [22]:
%page 0
        MOV A, [data]   ; A = 0xDE
        INC A            ; A = 0xDF
        MOV [data], A   ; записать обратно
        HLT
data:   DB 0xDE, 0xAD, 0xBE, 0xEF

CPU · HALTED · 3 steps · 5 cycles
  A=0xDF  B=0x00  C=0x00  D=0x00  SP=0xE7  DP=0x00  IP=0x08
  FLAGS: —
  FA=0x00000000  FB=0x00000000  FPCR=0x00  FPSR=0x00
  VA=0x0000  VB=0x0000  VC=0x0000  VM=0x0000  VL=0  VFPSR=0x00

Addr,+0,+1,+2,+3,+4,+5,+6,+7,+8,+9,+A,+B,+C,+D,+E,+F,ASCII
0000,02,00,09,12,00,04,09,00,00,DF,AD,BE,EF,00,00,00,················
0010,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0020,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0030,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0040,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0050,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0060,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0070,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0080,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0090,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················


## 4. FPU (arch=2)

`%arch 2` включает FPU-регистры в вывод (`FA`, `FB`, `FPCR`, `FPSR`).

In [3]:
%arch 2
; sqrt(2) * sqrt(2) = 2.0
        FMOV.H FHA, 2.0
        FSQRT.H FHA         ; FHA ≈ 1.414 (float16)
        FMOV.H FHB, FHA     ; FHB = sqrt(2)
        FMUL.H FHA, FHB     ; FHA ≈ 2.0
        HLT

CPU · HALTED · 4 steps · 7 cycles
  A=0x00  B=0x00  C=0x00  D=0x00  SP=0xE7  DP=0x00  IP=0x0C
  FLAGS: —
  FA=0x3DA84000  FB=0x00000000  FPCR=0x00  FPSR=0x10

### FPCR — режим округления; FPSR — sticky-исключения

`FSCFG A` записывает биты `[1:0]` из A в FPCR (0=round-nearest, 1=toward+∞, 2=toward−∞/zero).  
`FPSR` аккумулирует исключения: бит 2 = OF (overflow), бит 4 = NX (inexact).  
Переполнение float16 (max ≈ 65504): 60000 × 60000 → ∞, FPSR = 0x14 (OF + NX).

In [4]:
%arch 2
; FPCR = 2 (round toward zero), затем 1.7 → A=1 (truncate)
; FPSR.NX (inexact) устанавливается при преобразовании
        MOV A, 2
        FSCFG A             ; FPCR = 2 (round toward zero)
        FMOV.H FHA, 1.7     ; FHA = 1.7
        FFTOI.H A, FHA      ; A = 1 (truncated)
        FITOF.H FHB, A      ; FHB = 1.0
        HLT

CPU · HALTED · 5 steps · 7 cycles
  A=0x01  B=0x00  C=0x00  D=0x00  SP=0xE7  DP=0x00  IP=0x0F
  FLAGS: —
  FA=0x3C003ECD  FB=0x00000000  FPCR=0x02  FPSR=0x10

In [5]:
%arch 2
; float16 overflow: 60000 * 60000 → +inf, FPSR = 0x14 (OF + NX)
        FMOV.H FHA, 60000.0
        FMOV.H FHB, 60000.0
        FMUL.H FHA, FHB     ; FHA = +inf
        HLT

CPU · HALTED · 3 steps · 4 cycles
  A=0x00  B=0x00  C=0x00  D=0x00  SP=0xE7  DP=0x00  IP=0x0B
  FLAGS: —
  FA=0x7B537C00  FB=0x00000000  FPCR=0x00  FPSR=0x14

## 5. VU — скалярное произведение (arch=3)

`%arch 3` включает VU-регистры (`VA`, `VB`, `VC`, `VM`, `VL`, `VFPSR`).

`VMUL.U` + `VADD.U` — асинхронные векторные операции над 4 элементами uint8.
Результат: 2·10 + 3·20 + 1·30 + 4·40 = 270 mod 256 = **14**.

In [6]:
%arch 3
%page 1
%trace

@page 1
weights:  DB 2, 3, 1, 4
inputs:   DB 10, 20, 30, 40
product:  DB 0, 0, 0, 0
result:   DB 0

@page 0
        VSET VL, 4
        VSET VA, weights
        VSET VB, inputs
        VSET VC, product
        VMUL.U VC, VA, VB
        VSET VA, product
        VSET VC, result
        VADD.U VC, VA
        VWAIT
        HLT

CPU · HALTED · 9 steps · 9 cycles
  A=0x00  B=0x00  C=0x00  D=0x00  SP=0xE7  DP=0x00  IP=0x1F
  FLAGS: —
  FA=0x00000000  FB=0x00000000  FPCR=0x00  FPSR=0x00
  VA=0x010C  VB=0x0108  VC=0x010D  VM=0x0000  VL=4  VFPSR=0x04

Addr,+0,+1,+2,+3,+4,+5,+6,+7,+8,+9,+A,+B,+C,+D,+E,+F,ASCII
0100,02,03,01,04,0A,14,1E,28,14,3C,1E,A0,0E,00,00,00,·······(·<······
0110,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0120,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0130,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0140,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0150,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0160,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0170,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0180,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0190,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················


#,IP,Opcode,Operands,Changes,Cost,Fault
0,0,163,0x04 0x04 0x00,"IP=4, VL=4",1,
1,4,163,0x00 0x00 0x01,"IP=8, VA=256",1,
2,8,163,0x01 0x04 0x01,"IP=12, VB=260",1,
3,12,163,0x02 0x08 0x01,"IP=16, VC=264",1,
4,16,172,0x05 0x84,"IP=19, VA=260, VB=264, VC=268",1,
5,19,163,0x00 0x08 0x01,"IP=23, VA=264",1,
6,23,163,0x02 0x0C 0x01,IP=27,1,
7,27,170,0x1D 0x80,"IP=30, VA=268, VC=269",1,
8,30,169,—,IP=31,1,


### VU float16 (VMUL.H)

`VMUL.H` — поэлементное умножение векторов в формате float16.  
VA/VB/VC — 16-битные указатели на данные в памяти; VL — длина вектора; VFPSR — sticky FP-исключения VU.  
Результат: [1.0, 2.0, 3.0, 4.0] × [2.0, 2.0, 2.0, 2.0] = [2.0, 4.0, 6.0, 8.0] (float16 в странице 1).

In [7]:
%arch 3
%page 1

@page 1
a_vec:  DB 1.0_h, 2.0_h, 3.0_h, 4.0_h
b_vec:  DB 2.0_h, 2.0_h, 2.0_h, 2.0_h
result: DB 0, 0, 0, 0, 0, 0, 0, 0   ; 4 × float16 = 8 байт

@page 0
        VSET VL, 4
        VSET VA, a_vec
        VSET VB, b_vec
        VSET VC, result
        VMUL.H VC, VA, VB   ; result = [2.0, 4.0, 6.0, 8.0]
        VWAIT
        HLT

CPU · HALTED · 6 steps · 6 cycles
  A=0x00  B=0x00  C=0x00  D=0x00  SP=0xE7  DP=0x00  IP=0x14
  FLAGS: —
  FA=0x00000000  FB=0x00000000  FPCR=0x00  FPSR=0x00
  VA=0x0108  VB=0x0110  VC=0x0118  VM=0x0000  VL=4  VFPSR=0x00

Addr,+0,+1,+2,+3,+4,+5,+6,+7,+8,+9,+A,+B,+C,+D,+E,+F,ASCII
0100,00,3C,00,40,00,42,00,44,00,40,00,40,00,40,00,40,·<·@·B·D·@·@·@·@
0110,00,40,00,44,00,46,00,48,00,00,00,00,00,00,00,00,·@·D·F·H········
0120,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0130,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0140,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0150,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0160,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0170,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0180,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················
0190,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,00,················


## 6. FAULT

При нарушении (деление на ноль, выход за границу памяти и др.) CPU переходит в FAULT.
Статус окрашивается красным, флаг `F` устанавливается.

In [8]:
%arch 1
%trace
        MOV A, 42
        MOV B, 0
        DIV A, B    ; → FAULT(DIV_ZERO)
        HLT

AssemblerError: line 3: DIV: too many arguments
